In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Classical feedforward dan control flow (dynamic circuits)

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Dynamic circuits adalah alat yang powerful yang memungkinkan kamu mengukur qubit di tengah eksekusi quantum circuit, lalu melakukan operasi logika klasik di dalam circuit berdasarkan hasil pengukuran mid-circuit tersebut. Proses ini juga dikenal sebagai _classical feedforward_. Meskipun masih tahap awal dalam memahami cara terbaik memanfaatkan dynamic circuits, komunitas riset kuantum sudah mengidentifikasi sejumlah kasus penggunaan, seperti berikut:

* Persiapan quantum state yang efisien, seperti [GHZ state](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [W-state](https://arxiv.org/abs/2403.07604) (untuk informasi lebih lanjut tentang W-state, lihat juga ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), dan berbagai kelas [matrix product states](https://arxiv.org/abs/2404.16083)
* [Long-range entanglement yang efisien](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) antar qubit di chip yang sama menggunakan shallow circuits
* [Sampling IQP-like circuits](https://arxiv.org/pdf/2505.04705) yang efisien
## Pernyataan `if`
Pernyataan `if` digunakan untuk melakukan operasi secara kondisional berdasarkan nilai sebuah classical bit atau register.

Pada contoh di bawah, kita menerapkan Hadamard gate pada sebuah qubit dan mengukurnya. Jika hasilnya adalah 1, maka kita menerapkan X gate pada qubit tersebut, yang efeknya adalah membaliknya kembali ke state 0. Kita kemudian mengukur qubit itu lagi. Hasil pengukuran yang dihasilkan seharusnya 0 dengan probabilitas 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Pernyataan `with` bisa diberi target penugasan yang berupa context manager itu sendiri, yang bisa disimpan dan kemudian digunakan untuk membuat blok else, yang dieksekusi setiap kali isi blok `if` *tidak* dieksekusi.

Pada contoh di bawah, kita menginisialisasi register dengan dua qubit dan dua classical bit. Kita menerapkan Hadamard gate pada qubit pertama dan mengukurnya. Jika hasilnya adalah 1, maka kita menerapkan Hadamard gate pada qubit kedua; jika tidak, kita menerapkan X gate pada qubit kedua. Terakhir, kita juga mengukur qubit kedua.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Selain mengondisikan pada satu classical bit, kamu juga bisa mengondisikan pada nilai sebuah classical register yang terdiri dari beberapa bit.

Pada contoh di bawah, kita menerapkan Hadamard gates pada dua qubit dan mengukurnya. Jika hasilnya adalah `01`, yaitu qubit pertama adalah 1 dan qubit kedua adalah 0, maka kita menerapkan X gate pada qubit ketiga. Terakhir, kita mengukur qubit ketiga. Perlu dicatat bahwa demi kejelasan, kita memilih untuk menentukan state classical bit ketiga, yaitu 0, dalam kondisi `if`. Dalam gambar circuit, kondisi ditunjukkan oleh lingkaran pada classical bit yang dijadikan kondisi. Lingkaran solid menunjukkan pengondisian pada 1, sedangkan lingkaran bergaris menunjukkan pengondisian pada 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Ekspresi klasik
Modul ekspresi klasik Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) berisi representasi eksploratif dari operasi runtime pada nilai klasik selama eksekusi circuit. Karena keterbatasan hardware, hanya kondisi `QuantumCircuit.if_test()` yang saat ini didukung.

Contoh berikut menunjukkan bahwa kamu bisa menggunakan perhitungan paritas untuk membuat n-qubit GHZ state menggunakan dynamic circuits. Pertama, buat $n/2$ pasangan Bell pada qubit yang berdekatan. Kemudian, gabungkan pasangan-pasangan ini menggunakan satu lapisan CNOT gates di antara pasangan. Kamu kemudian mengukur target qubit dari semua CNOT gates sebelumnya dan me-reset setiap qubit yang diukur ke state $\vert 0 \rangle$. Kamu menerapkan $X$ ke setiap situs yang tidak diukur di mana paritas dari semua bit sebelumnya adalah ganjil. Terakhir, CNOT gates diterapkan pada qubit yang diukur untuk memulihkan entanglement yang hilang saat pengukuran.

Dalam perhitungan paritas, elemen pertama dari ekspresi yang dibangun melibatkan pengangkatan objek Python `mr[0]` ke sebuah node [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` digunakan untuk mengubah objek sembarang menjadi ekspresi klasik). Ini tidak diperlukan untuk `mr[1]` dan kemungkinan classical register berikutnya, karena keduanya adalah input untuk `expr.bit_xor`, dan pengangkatan yang diperlukan dilakukan secara otomatis dalam kasus tersebut. Ekspresi semacam itu bisa dibangun dalam loop dan konstruk lainnya.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

Because the example above used a single classical bit, there were only two possible cases, so you could have achieved the same result using an if-else statement. The switch case is mainly useful when branching on the value of a classical register composed of multiple bits. The following example shows how to construct a default case, which is executed if none of the preceding cases are. Note that in a switch statement, only one of the blocks are ever executed. There is no fallthrough.

The example below applies Hadamard gates to two qubits and measures them. If the result is either 00 or 11, apply a Z gate to the third qubit. If the result is 01, apply a Y gate. If none of the preceding cases matched, apply an X gate. Finally, measure the third qubit.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

<span id="store"></span>
### `Store`

Anda dapat menggunakan instruksi [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) untuk menyimpan hasil ekspresi klasik, jika ekspresi tersebut akan digunakan berulang kali. Operasi secara otomatis diparalelkan, sehingga kode Anda menjadi jauh lebih efisien saat runtime.

Misalnya, lebih alami dan lebih efisien saat runtime untuk menulis $B[0] \oplus B[1] \oplus B[2] \ldots$, di mana $B = \neg A$, daripada $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  Yang pertama menghitung negasi dalam satu langkah paralel sebelum rantai XOR, alih-alih mengevaluasi setiap negasi secara berurutan di dalam ekspresi.

Contoh lengkap:

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif)

## Langkah selanjutnya
> **Tip:** - Pelajari cara mengimplementasikan dynamic decoupling yang akurat menggunakan [stretch](/guides/stretch).
> - Gunakan [visualisasi jadwal circuit](/guides/qiskit-runtime-circuit-timing) untuk men-debug dan mengoptimalkan dynamic circuits kamu.
> - [Jalankan dynamic circuits](/guides/execute-dynamic-circuits).

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

## Classical expressions

The Qiskit classical expression module [`qiskit.circuit.classical`](/docs/api/qiskit/circuit_classical) contains an exploratory representation of runtime operations on classical values during circuit execution.

The following example shows that you can use the calculation of the parity to create an n-qubit GHZ state using dynamic circuits. First, generate $n/2$ Bell pairs on adjacent qubits. Then, glue these pairs together using a layer of CNOT gates in between pairs. You then measure the target qubit of all prior CNOT gates and reset each measured qubit to the state $\vert 0 \rangle$. You apply $X$ to every unmeasured site for which the parity of all preceding bits is odd. Finally, CNOT gates are applied to the measured qubits to re-establish the entanglement lost in the measurement.


In the parity calculation, the first element of the constructed expression involves lifting the Python object `mr[0]` to a [`Value`](/docs/api/qiskit/circuit_classical#value) node (`lift` is used to turn arbitrary objects into classical expressions). This is not necessary for `mr[1]` and the possible following classical register, as they are inputs to `expr.bit_xor`, and any necessary lifting is done automatically in these cases. Such expressions can be built up in loops and other constructs.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

<span id="store"></span>
### `Store`

You can use the [`store`](/docs/api/qiskit/circuit#store) instruction to save the result of a classical expression, if that expression will be used repeatedly. Operations are automatically parallelized, making your code significantly more efficient at runtime.

For example, it is more natural and more efficient at runtime to write $B[0] \oplus B[1] \oplus B[2] \ldots$, where $B = \neg A$, than $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  The former computes the negation in a single parallel step before the XOR chain, instead of evaluating each negation sequentially inside the expression.

Full example:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
- [Execute dynamic circuits](/docs/guides/execute-dynamic-circuits).
</Admonition>